In [ ]:
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.cache import TaskResultCache
from aereo.executors import LocalExecutor
from aereo.pipeline import ExtractionJob
from aereo.search_aws_goes import search_aws_goes

job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_goes19",
)

In [ ]:
from datetime import datetime, timezone

assets = search_aws_goes(
    collections={"ABI-L1b-RadF": ["C02"]},
    intersects="config/aoi/cordoba.geojson",
    start_datetime=datetime(2024, 1, 1, 15, 0, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 1, 15, 5, tzinfo=timezone.utc),
    satellites=["GOES-16"],
)
print(f"✓ Found {len(assets)} scenes")

In [ ]:
assets

In [ ]:
tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=5,
)
print(f"✓ Built {len(tasks)} tasks")

In [ ]:
artifacts = job.execute(
    tasks,
    executor=LocalExecutor(workers=2, cache=TaskResultCache()),
)
print(f"✓ Extracted {len(artifacts)} artifacts")

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, cmap="viridis")